# Bundestagswahl 2025: Wählerwanderung

Estimated voter movement from the 2021 to the 2025 Bundestag election, using the aggregate edges in `bundestagswahl_2025_voter_migration_sankey_data.csv`.

> **Method note:** Migration values are model-based estimates transcribed from tagesschau / infratest dimap graphics, not exact individual ballot tracking. Wahlrecht.de is used for election-result context and does not provide the complete voter-migration matrix.

The CSV contains the `Sankey_Input` worksheet data. The original multi-sheet workbook is retained at the project root for context. The export cell writes the responsive HTML page to `p00/output`.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


def find_project_root(start: Path = Path.cwd()) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if candidate.name == "p00":
            return candidate
    raise RuntimeError(f"Could not find a parent folder named 'p00' from {start.resolve()}")


project_root = find_project_root()
csv_path = project_root / "bundestagswahl_2025_voter_migration_sankey_data.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Migration CSV not found: {csv_path}")

data = pd.read_csv(csv_path, encoding="utf-8")
required_columns = {"source_node", "target_node", "value_voters", "color_hex", "label", "source_url"}
missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise ValueError(f"Migration CSV is missing required columns: {sorted(missing_columns)}")

data = data.dropna(subset=["source_node", "target_node", "value_voters"]).copy()
data["value_voters"] = pd.to_numeric(data["value_voters"], errors="raise")
if data.empty or (data["value_voters"] <= 0).any():
    raise ValueError("Migration CSV must contain positive voter estimates.")

print(f"Loaded {len(data)} estimated flows ({data['value_voters'].sum():,.0f} voters represented).")

In [ ]:
party_colors = {
    "Union": "#D4D8DE",
    "SPD": "#E31B23",
    "AfD": "#009EE0",
    "Grüne": "#64B914",
    "FDP": "#FFCC00",
    "Linke": "#BE3075",
    "BSW": "#47B9D4",
    "Nichtwähler": "#8B949E",
}
fallback_color = "#AEB8C4"


def color_for_node(node_label: str) -> str:
    party = node_label.split(": ", maxsplit=1)[-1]
    return party_colors.get(party, fallback_color)


def rgba(hex_color: str, alpha: float = 0.32) -> str:
    hex_color = hex_color.lstrip("#")
    red, green, blue = (int(hex_color[index:index + 2], 16) for index in (0, 2, 4))
    return f"rgba({red}, {green}, {blue}, {alpha})"


source_totals = data.groupby("source_node")["value_voters"].sum().sort_values(ascending=False)
target_totals = data.groupby("target_node")["value_voters"].sum().sort_values(ascending=False)
source_nodes = source_totals.index.tolist()
target_nodes = target_totals.index.tolist()
node_labels = source_nodes + target_nodes
node_index = {label: index for index, label in enumerate(node_labels)}
node_x = [0.01] * len(source_nodes) + [0.99] * len(target_nodes)
node_y = [(index + 0.5) / len(source_nodes) for index in range(len(source_nodes))]
node_y += [(index + 0.5) / len(target_nodes) for index in range(len(target_nodes))]

link_customdata = [
    [row.label, f"{int(row.value_voters):,}", row.source_url]
    for row in data.itertuples(index=False)
]
link_colors = [rgba(color_for_node(source)) for source in data["source_node"]]

fig = go.Figure(
    go.Sankey(
        arrangement="freeform",
        node=dict(
            label=node_labels,
            x=node_x,
            y=node_y,
            color=[color_for_node(label) for label in node_labels],
            pad=22,
            thickness=18,
            line=dict(color="rgba(245,247,250,0.35)", width=0.7),
            hovertemplate="%{label}<br>Estimated total: %{value:,.0f} voters<extra></extra>",
        ),
        link=dict(
            source=[node_index[label] for label in data["source_node"]],
            target=[node_index[label] for label in data["target_node"]],
            value=data["value_voters"].tolist(),
            color=link_colors,
            customdata=link_customdata,
            hovertemplate="%{customdata[0]}<br>%{customdata[1]} estimated voters<br><sup>%{customdata[2]}</sup><extra></extra>",
        ),
    )
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(family="system-ui, sans-serif", size=13, color="#f5f7fa"),
    width=None,
    height=820,
    autosize=True,
    margin=dict(l=18, r=18, t=28, b=22),
)
fig.show()

In [ ]:
project_root = find_project_root()
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "20261008_bundestagswahl_sankey.html"

plot_html = fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    config={"responsive": True, "displaylogo": False},
)

html = f"""<!DOCTYPE html>
<html lang="de">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <meta name="theme-color" content="#0d1117">
  <title>Wählerwanderung 2021–2025 | phey.app</title>
  <style>
    * {{ box-sizing: border-box; }}
    html, body {{ margin: 0; min-height: 100%; background: #0d1117; color: #f5f7fa; }}
    body {{ font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
    .plot-page {{ min-height: 100vh; padding: clamp(0.75rem, 2vw, 1.5rem); }}
    .plot-wrap {{ width: 100%; margin: 0 auto; }}
    .back-link {{ display: inline-flex; margin-bottom: 1rem; color: #c7d0dd; text-decoration: none; }}
    .back-link:hover {{ color: #ffffff; }}
    h1 {{ margin: 0; font-size: clamp(1.55rem, 3vw, 2.25rem); font-weight: 650; }}
    .subtitle {{ max-width: 70rem; margin: 0.45rem 0 0; color: #c7d0dd; line-height: 1.5; }}
    .method-note {{ max-width: 85rem; margin: 0.75rem 0 0.25rem; padding-left: 0.8rem; border-left: 3px solid #d4a72c; color: #aeb8c4; line-height: 1.5; }}
    .sources {{ margin: 0.75rem 0 1.5rem; color: #8b949e; font-size: 0.9rem; }}
    .sources a {{ color: #8ecae6; }}
    @media (max-width: 640px) {{ .method-note {{ font-size: 0.92rem; }} }}
  </style>
</head>
<body>
  <main class="plot-page">
    <div class="plot-wrap">
      <a class="back-link" href="../index.html">← Back to phey.app</a>
      <header>
        <h1>Wählerwanderung zur Bundestagswahl 2025</h1>
        <p class="subtitle">Estimated voter movement from the 2021 election choice to the 2025 election choice. Flow width represents the estimated number of voters.</p>
        <p class="method-note"><strong>Method:</strong> These are model-based estimates transcribed from tagesschau / infratest dimap voter-migration graphics, not exact individual ballot tracking. Wahlrecht.de is used for election-result context; it does not provide the complete migration matrix.</p>
        <p class="sources">Sources: <a href="https://www.tagesschau.de/wahl/archiv/2025-02-23-BT-DE/analyse-wanderung.shtml" target="_blank" rel="noopener noreferrer">tagesschau / infratest dimap</a> for estimated migration; <a href="https://www.wahlrecht.de/ergebnisse/bundestag.htm" target="_blank" rel="noopener noreferrer">Wahlrecht.de</a> for election results context.</p>
      </header>
      {plot_html}
    </div>
  </main>
</body>
</html>
"""

output_file.write_text(html, encoding="utf-8")
print(f"Exported Bundestag Sankey to: {output_file.resolve()}")